In [1]:
import os
import json
import math
import copy
import random
from pathlib import Path

import numpy as np
import pandas as pd
import tifffile as tiff
from PIL import Image

import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from skimage.transform import resize
from skimage.measure import label as sk_label

from cellpose import models

In [2]:
PROJECT_ROOT = Path(r"C:\Users\wangy\Downloads\cnn")
TRAIN_IMG_DIR = PROJECT_ROOT / "Training" / "images"
TRAIN_GT_DIR  = PROJECT_ROOT / "Training" / "ground_truth"
OUT_DIR       = PROJECT_ROOT / "outputs" / "train_cnn_pipeline"

ORACLE_DIR    = OUT_DIR / "oracle_masks"
MODEL_DIR     = OUT_DIR / "model"
FIG_DIR       = OUT_DIR / "figures"

for d in [OUT_DIR, ORACLE_DIR, MODEL_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

IMAGE_EXTS = {".png", ".bmp", ".tif", ".tiff", ".jpg", ".jpeg"}

USE_GPU = True
DEVICE = torch.device("cuda" if USE_GPU and torch.cuda.is_available() else "cpu")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("DEVICE:", DEVICE)
print("TRAIN_IMG_DIR exists:", TRAIN_IMG_DIR.exists())
print("TRAIN_GT_DIR exists:", TRAIN_GT_DIR.exists())

DEVICE: cuda
TRAIN_IMG_DIR exists: True
TRAIN_GT_DIR exists: True


In [3]:
IMG_SIZE = 128
BATCH_SIZE = 16
MAX_EPOCHS = 40
PATIENCE = 8
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

MODEL_TYPE = "cyto3"
CHANNELS = [0, 0]

DIAMETERS = [15, 20, 25, 30, 35]
CELLPROB_THRESHOLDS = [-2, 0, 2]
FLOW_THRESHOLDS = [0.2, 0.4, 0.6]

MODEL_BUNDLE_PATH = MODEL_DIR / "cnn.pt"
ORACLE_SWEEP_CSV = OUT_DIR / "oracle_sweep_results.csv"
ORACLE_BEST_CSV = OUT_DIR / "oracle_best_per_image.csv"
TRAIN_HISTORY_CSV = OUT_DIR / "cnn_training_history.csv"

In [4]:
def list_images(folder):
    folder = Path(folder)
    files = [p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS]
    return sorted(files)

def image_key_from_path(p):
    p = Path(p)
    stem = p.stem
    if stem.endswith("_label"):
        stem = stem[:-6]
    return stem

def read_image_any(path):
    path = Path(path)
    ext = path.suffix.lower()
    if ext in {".tif", ".tiff"}:
        arr = tiff.imread(str(path))
    else:
        arr = np.array(Image.open(path))

    arr = np.asarray(arr)

    if arr.ndim == 3:
        if arr.shape[-1] in (3, 4):
            arr = arr[..., :3].mean(axis=-1)
        else:
            arr = arr[0]

    arr = np.asarray(arr, dtype=np.float32)
    return arr

def read_mask_any(path):
    arr = read_image_any(path)
    arr = np.asarray(arr)

    if arr.ndim != 2:
        raise ValueError(f"Mask must be 2D after sanitization, got {arr.shape} for {path}")

    if np.issubdtype(arr.dtype, np.floating):
        arr = np.rint(arr)

    arr = arr.astype(np.int32)

    if arr.max() <= 1:
        arr = sk_label(arr > 0)

    return arr

def normalize_image(img):
    img = np.asarray(img, dtype=np.float32)
    lo = np.percentile(img, 1)
    hi = np.percentile(img, 99)
    if hi <= lo:
        hi = img.max() if img.max() > lo else lo + 1.0
    img = (img - lo) / (hi - lo)
    img = np.clip(img, 0, 1)
    return img.astype(np.float32)

def resize_image_for_cnn(img, out_size=IMG_SIZE):
    img = normalize_image(img)
    if img.shape != (out_size, out_size):
        img = resize(
            img,
            (out_size, out_size),
            preserve_range=True,
            anti_aliasing=True
        ).astype(np.float32)
    return img

def resize_mask_to_shape(mask, shape_hw):
    if mask.shape == shape_hw:
        return mask.astype(np.int32)
    mask2 = resize(
        mask.astype(np.float32),
        shape_hw,
        order=0,
        preserve_range=True,
        anti_aliasing=False
    )
    mask2 = np.rint(mask2).astype(np.int32)
    if mask2.max() <= 1:
        mask2 = sk_label(mask2 > 0)
    return mask2

In [5]:
def binary_iou(gt, pred):
    gt_b = (gt > 0)
    pr_b = (pred > 0)
    inter = np.logical_and(gt_b, pr_b).sum()
    union = np.logical_or(gt_b, pr_b).sum()
    if union == 0:
        return 1.0
    return inter / union

def binary_dice(gt, pred):
    gt_b = (gt > 0)
    pr_b = (pred > 0)
    inter = np.logical_and(gt_b, pr_b).sum()
    denom = gt_b.sum() + pr_b.sum()
    if denom == 0:
        return 1.0
    return 2.0 * inter / denom

def overlay_boundaries(image, mask):
    image = normalize_image(image)
    rgb = np.stack([image, image, image], axis=-1)

    mask_b = mask > 0
    edges = np.zeros_like(mask_b, dtype=bool)
    edges[:-1, :] |= (mask_b[:-1, :] != mask_b[1:, :])
    edges[1:,  :] |= (mask_b[:-1, :] != mask_b[1:, :])
    edges[:, :-1] |= (mask_b[:, :-1] != mask_b[:, 1:])
    edges[:, 1:]  |= (mask_b[:, :-1] != mask_b[:, 1:])

    rgb[edges, 0] = 1.0
    rgb[edges, 1] = 0.0
    rgb[edges, 2] = 0.0
    return rgb

In [6]:
train_images = list_images(TRAIN_IMG_DIR)
train_masks  = list_images(TRAIN_GT_DIR)

img_map = {image_key_from_path(p): p for p in train_images}
gt_map  = {image_key_from_path(p): p for p in train_masks}

common_keys = sorted(set(img_map) & set(gt_map))
missing_img = sorted(set(gt_map) - set(img_map))
missing_gt  = sorted(set(img_map) - set(gt_map))

print("matched pairs:", len(common_keys))
print("missing images for GT:", len(missing_img))
print("missing GT for images:", len(missing_gt))

pairs = pd.DataFrame({
    "key": common_keys,
    "img_path": [str(img_map[k]) for k in common_keys],
    "gt_path": [str(gt_map[k]) for k in common_keys],
})

display(pairs.head())

matched pairs: 1000
missing images for GT: 0
missing GT for images: 0


,key,img_path,gt_path
0,cell_00001,C:\Users\wangy\Downloads\cnn\Training\images\c...,C:\Users\wangy\Downloads\cnn\Training\ground_t...
1,cell_00002,C:\Users\wangy\Downloads\cnn\Training\images\c...,C:\Users\wangy\Downloads\cnn\Training\ground_t...
2,cell_00003,C:\Users\wangy\Downloads\cnn\Training\images\c...,C:\Users\wangy\Downloads\cnn\Training\ground_t...
3,cell_00004,C:\Users\wangy\Downloads\cnn\Training\images\c...,C:\Users\wangy\Downloads\cnn\Training\ground_t...
4,cell_00005,C:\Users\wangy\Downloads\cnn\Training\images\c...,C:\Users\wangy\Downloads\cnn\Training\ground_t...


In [7]:
param_rows = []
combo_id = 0
for d in DIAMETERS:
    for cp in CELLPROB_THRESHOLDS:
        for fl in FLOW_THRESHOLDS:
            param_rows.append({
                "combo_id": combo_id,
                "diameter": float(d),
                "cellprob_threshold": float(cp),
                "flow_threshold": float(fl),
            })
            combo_id += 1

combo_table = pd.DataFrame(param_rows)
display(combo_table.head())
print("num combos:", len(combo_table))

,combo_id,diameter,cellprob_threshold,flow_threshold
0,0,15.0,-2.0,0.2
1,1,15.0,-2.0,0.4
2,2,15.0,-2.0,0.6
3,3,15.0,0.0,0.2
4,4,15.0,0.0,0.4


num combos: 45


In [8]:
cp_model = models.CellposeModel(
    gpu=(DEVICE.type == "cuda"),
    model_type=MODEL_TYPE
)
print(cp_model)

model_type argument is not used in v4.0.1+. Ignoring this argument...


In [14]:
RUN_SWEEP = True
SWEEP_MAX_IMAGES = 100

In [ ]:
from tqdm.auto import tqdm
import itertools

all_sweep_rows = []
best_rows = []

if RUN_SWEEP:
    combos = list(itertools.product(
        DIAMETERS,
        CELLPROB_THRESHOLDS,
        FLOW_THRESHOLDS
    ))

    pairs_for_sweep = pairs.copy()
    if SWEEP_MAX_IMAGES is not None:
        pairs_for_sweep = pairs_for_sweep.iloc[: int(SWEEP_MAX_IMAGES)].copy()

    total_runs = len(combos) * len(pairs_for_sweep)
    print(
        f"Sweep: {len(combos)} param combos × {len(pairs_for_sweep)} image(s)"
        f" = {total_runs} total Cellpose runs"
    )

    cp_model = models.CellposeModel(
        gpu=(DEVICE.type == "cuda"),
        model_type=MODEL_TYPE
    )

    pbar = tqdm(total=total_runs, desc="oracle sweep", unit="run")

    for _, row in pairs_for_sweep.iterrows():
        key = row["key"]
        img_path = Path(row["img_path"])
        gt_path = Path(row["gt_path"])

        img_raw = read_image_any(img_path)
        img_proc = normalize_image(img_raw)

        gt_mask_cached = read_mask_any(gt_path)
        if gt_mask_cached.shape != img_proc.shape:
            gt_mask_cached = resize_mask_to_shape(gt_mask_cached, img_proc.shape)

        best_dice = -1.0
        best_entry = None
        best_mask = None

        for diameter, cp, ft in combos:
            masks, flows, styles = cp_model.eval(
                img_proc,
                channels=CHANNELS,
                diameter=float(diameter),
                cellprob_threshold=float(cp),
                flow_threshold=float(ft),
            )

            pred_mask = np.asarray(masks)
            if pred_mask.shape != img_proc.shape:
                pred_mask = resize_mask_to_shape(pred_mask, img_proc.shape)

            dice = binary_dice(gt_mask_cached, pred_mask)
            iou = binary_iou(gt_mask_cached, pred_mask)

            rec = {
                "image_key": key,
                "img_path": str(img_path),
                "gt_path": str(gt_path),
                "diameter": float(diameter),
                "cellprob_threshold": float(cp),
                "flow_threshold": float(ft),
                "dice": float(dice),
                "iou": float(iou),
            }
            all_sweep_rows.append(rec)

            if dice > best_dice:
                best_dice = float(dice)
                best_entry = rec.copy()
                best_mask = pred_mask.copy()

            pbar.update(1)

        best_rows.append(best_entry)

        tiff.imwrite(
            str(ORACLE_DIR / f"{key}_oracle_mask.tif"),
            best_mask.astype(np.int32)
        )

    pbar.close()

    sweep_df = pd.DataFrame(all_sweep_rows)

    combo_table_tmp = []
    combo_id = 0
    for d in DIAMETERS:
        for cp in CELLPROB_THRESHOLDS:
            for ft in FLOW_THRESHOLDS:
                combo_table_tmp.append({
                    "combo_id": combo_id,
                    "diameter": float(d),
                    "cellprob_threshold": float(cp),
                    "flow_threshold": float(ft),
                })
                combo_id += 1
    combo_table = pd.DataFrame(combo_table_tmp)

    best_df = pd.DataFrame(best_rows).merge(
        combo_table,
        on=["diameter", "cellprob_threshold", "flow_threshold"],
        how="left"
    )

    sweep_df.to_csv(ORACLE_SWEEP_CSV, index=False)
    best_df.to_csv(ORACLE_BEST_CSV, index=False)

    print("Saved:", ORACLE_SWEEP_CSV)
    print("Saved:", ORACLE_BEST_CSV)
    display(best_df.head())

else:
    sweep_df = pd.DataFrame()
    best_df = pd.DataFrame()

model_type argument is not used in v4.0.1+. Ignoring this argument...


Sweep: 45 param combos × 100 image(s) = 4500 total Cellpose runs


oracle sweep:   0%|          | 0/4500 [00:00<?, ?run/s]

channels deprecated in v4.0.1+. If data contain more than 3 channels, only the first 3 channels will be used
channels deprecated in v4.0.1+. If data contain more than 3 channels, only the first 3 channels will be used


In [ ]:
train_df = best_df.copy()
train_df["combo_id"] = train_df["combo_id"].astype(int)

display(train_df[["image_key", "combo_id", "diameter", "cellprob_threshold", "flow_threshold", "dice"]].head())
print("num training images:", len(train_df))
print("num unique combo labels used:", train_df["combo_id"].nunique())

In [ ]:
class ParamComboDataset(Dataset):
    def __init__(self, df, img_size=IMG_SIZE):
        self.df = df.reset_index(drop=True).copy()
        self.img_size = img_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = read_image_any(row["img_path"])
        img = resize_image_for_cnn(img, self.img_size)
        x = torch.tensor(img[None, :, :], dtype=torch.float32)
        y = torch.tensor(int(row["combo_id"]), dtype=torch.long)
        return x, y

In [ ]:
train_split, val_split = train_test_split(
    train_df,
    test_size=0.2,
    random_state=SEED,
    stratify=train_df["combo_id"] if train_df["combo_id"].nunique() > 1 else None
)

print("train:", len(train_split))
print("val:", len(val_split))

train_ds = ParamComboDataset(train_split, img_size=IMG_SIZE)
val_ds   = ParamComboDataset(val_split, img_size=IMG_SIZE)

train_loader = DataLoader(
    train_ds,
    batch_size=min(BATCH_SIZE, len(train_ds)),
    shuffle=True,
    num_workers=0,
    pin_memory=(DEVICE.type == "cuda"),
)

val_loader = DataLoader(
    val_ds,
    batch_size=min(BATCH_SIZE, len(val_ds)),
    shuffle=False,
    num_workers=0,
    pin_memory=(DEVICE.type == "cuda"),
)

In [ ]:
class ParamComboCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

In [ ]:
num_classes = len(combo_table)
model = ParamComboCNN(num_classes=num_classes).to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

best_state = copy.deepcopy(model.state_dict())
best_val_loss = np.inf
best_epoch = -1
wait = 0
history = []

In [ ]:
for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    train_losses = []
    train_correct = 0
    train_total = 0

    for xb, yb in train_loader:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True)

        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        train_losses.append(float(loss.detach().cpu().item()))
        preds = logits.argmax(dim=1)
        train_correct += int((preds == yb).sum().item())
        train_total += int(yb.numel())

    train_loss = float(np.mean(train_losses)) if train_losses else np.nan
    train_acc = train_correct / train_total if train_total else np.nan

    model.eval()
    val_losses = []
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            logits = model(xb)
            loss = criterion(logits, yb)

            val_losses.append(float(loss.detach().cpu().item()))
            preds = logits.argmax(dim=1)
            val_correct += int((preds == yb).sum().item())
            val_total += int(yb.numel())

    val_loss = float(np.mean(val_losses)) if val_losses else np.nan
    val_acc = val_correct / val_total if val_total else np.nan

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc,
    })

    print(f"epoch {epoch:03d}  train_loss={train_loss:.4f}  train_acc={train_acc:.4f}  val_loss={val_loss:.4f}  val_acc={val_acc:.4f}")

    if val_loss < best_val_loss - 1e-6:
        best_val_loss = val_loss
        best_epoch = epoch
        best_state = copy.deepcopy(model.state_dict())
        wait = 0
    else:
        wait += 1
        if wait >= PATIENCE:
            print("early stopping")
            break

model.load_state_dict(best_state)
history_df = pd.DataFrame(history)
history_df.to_csv(TRAIN_HISTORY_CSV, index=False)

print("best_epoch:", best_epoch)
print("best_val_loss:", best_val_loss)
print("saved:", TRAIN_HISTORY_CSV)

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history_df["epoch"], history_df["train_loss"], marker="o", label="train_loss")
plt.plot(history_df["epoch"], history_df["val_loss"], marker="s", label="val_loss")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("training history")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "training_loss.png", dpi=180, bbox_inches="tight")
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(history_df["epoch"], history_df["train_acc"], marker="o", label="train_acc")
plt.plot(history_df["epoch"], history_df["val_acc"], marker="s", label="val_acc")
plt.xlabel("epoch")
plt.ylabel("accuracy")
plt.title("training accuracy")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "training_accuracy.png", dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
payload = {
    "state_dict": model.state_dict(),
    "img_size": IMG_SIZE,
    "model_type": MODEL_TYPE,
    "channels": CHANNELS,
    "combo_table": combo_table.to_dict(orient="records"),
    "best_epoch": int(best_epoch),
    "best_val_loss": float(best_val_loss),
}

torch.save(payload, MODEL_BUNDLE_PATH)
print("saved model bundle:", MODEL_BUNDLE_PATH)

In [ ]:
def predict_combo_from_image(model, combo_table_df, image_path, img_size=IMG_SIZE, device=DEVICE):
    img = read_image_any(image_path)
    x = resize_image_for_cnn(img, img_size)
    x = torch.tensor(x[None, None, :, :], dtype=torch.float32).to(device)

    model.eval()
    with torch.no_grad():
        logits = model(x)
        pred_class = int(logits.argmax(dim=1).item())

    combo = combo_table_df.loc[combo_table_df["combo_id"] == pred_class].iloc[0].to_dict()
    return pred_class, combo

val_preview = val_split.reset_index(drop=True).copy()
preview_rows = []

for i in range(min(8, len(val_preview))):
    row = val_preview.iloc[i]
    pred_class, combo = predict_combo_from_image(model, combo_table, row["img_path"])
    preview_rows.append({
        "image_key": row["image_key"],
        "true_combo_id": int(row["combo_id"]),
        "pred_combo_id": pred_class,
        "pred_diameter": combo["diameter"],
        "pred_cellprob_threshold": combo["cellprob_threshold"],
        "pred_flow_threshold": combo["flow_threshold"],
    })

preview_df = pd.DataFrame(preview_rows)
display(preview_df)